# MAIW v2 — Getting Started

**Multi-Agent Intelligent Warehouse — v2 Governed Agentic Operations**

This notebook follows the README Quickstart (steps 2–17):

1. Validate environment (Python, Node.js, GPU)
2. Install dependencies and all MAIW packages
3. Configure API keys and model endpoints
4. Warehouse World (DataPack) — validate the committed pack
5. Start backend API server
6. Start frontend UI
7. Open WORLD view (operational graph)
8. Activate the Wave 17 scenario (`labor_constraint_wave_risk`)
9. Ask Copilot: "What is the current wave risk?" (ASK)
10. Ask Copilot: "What should I do about the labor shortage?" (ANALYZE)
11. Inspect the OperationalContextSnapshot
12. Run the SOP-driven agent (OperationsCoordinationAgent → `wave_risk_resolution` SOP)
13. Review the RecommendedAction
14. Execute the governed action (Copilot ACT, then `POST /api/v1/demo/approve`)
15. Observe the LIVE outcome
16. Ask Copilot: "Did it work?" (OBSERVE_OUTCOME)
17. Open the Model Gateway Evaluation Lab

---

> **Prerequisites:** Python 3.11+, Node.js 20+, NVIDIA API key (`nvapi-...`)
>
> **Run the notebook from the repo root or `notebooks/`, inside the virtual environment you use for MAIW.**
>
> **No database required:** Demo Mode uses the local DataPack — no PostgreSQL, Redis, Milvus, or Kafka.
>
> The API base URL defaults to `http://localhost:8001`; override it with the `MAIW_API_BASE` environment variable.

## Step 1 — Validate Environment

In [ ]:
import subprocess
import sys

print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 11), "Python 3.11+ required"

# Check Node.js (frontend)
try:
    result = subprocess.run(["node", "--version"], capture_output=True, text=True)
    print(f"Node.js version: {result.stdout.strip()}")
except FileNotFoundError:
    print("WARNING: Node.js not found — frontend UI will not be available (Node.js 20+ recommended)")

# Check GPU (optional — only needed for a self-hosted NIM)
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    if gpu.returncode == 0:
        print(f"GPU: {gpu.stdout.strip()}")
    else:
        print("No GPU detected — using NVIDIA-hosted inference (requires NVIDIA API key)")
except FileNotFoundError:
    print("No GPU detected — using NVIDIA-hosted inference (requires NVIDIA API key)")

print("\nEnvironment check complete.")

## Step 2 — Install Dependencies and MAIW Packages

Same as README Quickstart step 2: install `requirements.txt`, then **all ten** MAIW packages
in **one** editable `pip install` command so the local packages satisfy each other's dependencies.
Run once after cloning, inside an activated virtual environment
(`python3 -m venv .venv && source .venv/bin/activate`).

In [ ]:
import pathlib
import subprocess
import sys

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    # Notebook may be running from notebooks/
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "packages").exists(), f"Could not find packages/ — run from the repo root. cwd={pathlib.Path.cwd()}"
print(f"Repo root: {REPO_ROOT}")

PACKAGES = [
    "packages/maiw-contracts",
    "packages/maiw-mcp",
    "packages/maiw-state",
    "packages/maiw-models",
    "packages/maiw-skills",
    "packages/maiw-decision",
    "packages/maiw-execution",
    "packages/maiw-agents",
    "packages/maiw-world",
    "apps/api",
]

steps = [
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    [sys.executable, "-m", "pip", "install", "-q"] + [arg for pkg in PACKAGES for arg in ("-e", pkg)],
]
for cmd in steps:
    print("$ " + " ".join(cmd[2:]))
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("pip install failed — see output above")

print("\nPackage install complete.")

## Step 3 — Configure API Keys and Model Endpoints

Copy `.env.example` to `.env` and set `NVIDIA_API_KEY`.
For a self-hosted NIM set `MAIW_NIM_BASE_URL` (and `MAIW_NIM_API_KEY` if needed), and enable only the
roles that endpoint serves with `NEMOTRON_<ROLE>_ENABLED` — see `docs/developer/MODEL_DEPLOYMENT.md`.
The backend reads `.env` itself; this cell only checks it.

In [ ]:
import os
import pathlib

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    REPO_ROOT = REPO_ROOT.parent

env_file = REPO_ROOT / ".env"
env_example = REPO_ROOT / ".env.example"

if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ.setdefault(key.strip(), val.split(" #")[0].strip())
    print(f"Loaded {env_file}")
else:
    print(f"No .env found at {env_file}")
    if env_example.exists():
        print(f"Copy {env_example} to {env_file} and set NVIDIA_API_KEY")

api_key = os.environ.get("NVIDIA_API_KEY", "")
if api_key.startswith("nvapi-"):
    print("NVIDIA_API_KEY: set")  # never print any part of the key
else:
    print("WARNING: NVIDIA_API_KEY not set or invalid — hosted inference will not work")

print(f"MAIW_NIM_BASE_URL: {os.environ.get('MAIW_NIM_BASE_URL') or '(not set — NVIDIA hosted endpoint)'}")
print(f"MAIW_AGENT_RUNTIME: {os.environ.get('MAIW_AGENT_RUNTIME', 'deterministic (default)')}")

## Step 4 — Warehouse World (DataPack)

The canonical DC-47 DataPack is committed at `data/worlds/dc47-demo-v1` (seed 42), so generation is optional.
To regenerate it: `python -m maiw_world generate --overwrite`.
This cell validates the committed pack (manifest, file checksums, semantic checksum, graph validity)
and inspects the Wave 17 entity.

In [ ]:
import pathlib
import subprocess
import sys

from maiw_world.datapack import WarehouseDataPack

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "packages").exists():
    REPO_ROOT = REPO_ROOT.parent
PACK_DIR = REPO_ROOT / "data" / "worlds" / "dc47-demo-v1"
assert (PACK_DIR / "manifest.json").exists(), f"DataPack not found at {PACK_DIR}"

manifest = WarehouseDataPack.read_manifest(PACK_DIR)
for key in ("dataset_id", "warehouse_id", "seed", "entity_count", "edge_count", "event_count", "semantic_checksum"):
    print(f"{key:18} {manifest.get(key)}")
print()

for cmd in (["validate", str(PACK_DIR)], ["inspect", str(PACK_DIR), "--entity", "wave-017"]):
    result = subprocess.run([sys.executable, "-m", "maiw_world", *cmd], cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f"python -m maiw_world {cmd[0]} failed")

## Steps 5-6 — Start Backend and Frontend

Run these in separate terminals from the repo root (not inside the notebook).

**Backend (step 5)** — development profile with in-memory procedure state:
```bash
MAIW_DEMO_MODE=true MAIW_PERSISTENCE_MODE=memory uvicorn maiw_api.app:app --reload --port 8001
```

**Frontend (step 6):**
```bash
cd src/ui/web && npm install && npm start
# Runs at http://localhost:3001 (bound to 127.0.0.1)
```

The UI opens on **OPERATIONS** (`/demo`). Other tabs: **WORLD** (operational graph), **RELIABILITY**,
**MODELS** (Model Gateway; the Lab is at `/models/lab`), **CAPABILITIES**, **ACTIVITY**, **DEPLOYMENT**.
The Copilot opens as a drawer in OPERATIONS.

The next cell defines a small `api()` helper used by every later request cell — run it first.

In [ ]:
import json
import os
import urllib.error
import urllib.request

API_BASE = os.environ.get("MAIW_API_BASE", "http://localhost:8001").rstrip("/")


def api(method, path, body=None, timeout=30):
    """Call the MAIW API. Returns parsed JSON, or None after printing a clear error."""
    data = json.dumps(body).encode() if body is not None else None
    headers = {"Content-Type": "application/json"} if data is not None else {}
    req = urllib.request.Request(f"{API_BASE}{path}", data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read()
            return json.loads(raw) if raw else None
    except urllib.error.HTTPError as exc:
        print(f"HTTP {exc.code} from {method} {path}: {exc.read().decode(errors='replace')[:800]}")
    except urllib.error.URLError as exc:
        print(f"Cannot reach {API_BASE}{path}: {exc.reason} — is the backend running (step 5)?")
    except TimeoutError:
        print(f"{method} {path} timed out after {timeout}s")
    return None


live = api("GET", "/api/v1/live")
if live:
    print(f"Liveness: {live.get('status')}  version={live.get('version')}")

# Readiness returns HTTP 503 with the failing checks when the profile is not ready.
ready = api("GET", "/api/v1/ready")
if ready:
    print(f"Readiness: {ready.get('status')}  profile={ready.get('profile')}")

## Step 7 — Open WORLD View

In the UI: **WORLD** tab (`http://localhost:3001/world`). The same data via the API:

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

summary = api("GET", "/api/v1/world/summary")
if summary:
    dp, graph, scen, rt = summary["datapack"], summary["graph"], summary["scenario"], summary["runtime"]
    print(f"DataPack:  {dp['dataset_id']} ({dp['warehouse_id']}, seed={dp['seed']}, loaded={dp['loaded']})")
    print(f"Checksum:  {dp.get('semantic_checksum')}")
    print(f"Graph:     {graph['total_entities']} entities, {graph['total_relationships']} relationships, {graph['event_count']} events")
    print(f"Entities:  {graph['entity_counts']}")
    print(f"Scenario:  active={scen['active']} name={scen.get('name')}")
    print(f"Runtime:   {rt['status']} clock={rt.get('clock_iso')}")

## Step 8 — Activate the Wave 17 Scenario

The Wave 17 at-risk scenario is `labor_constraint_wave_risk` (understaffed shift, wave at risk).
In the UI: OPERATIONS → select the scenario → **START**.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

SCENARIO = "labor_constraint_wave_risk"

result = api("POST", f"/api/v1/demo/scenario/{SCENARIO}/start")
if result:
    status = result["status"]
    print(f"Scenario started: {result['scenario']}  active={status['active']}")
    print(f"Pending approvals: {len(status.get('pending_approvals', []))}")
    print(f"KPIs: {status.get('current_kpis')}")

## Step 9 — Ask Copilot (ASK)

Copilot answers are grounded in the current warehouse state; every grounded turn captures an
`OperationalContextSnapshot`. Omit `conversation_id` on the first turn and reuse the returned one.
Requires a valid `NVIDIA_API_KEY` on the backend (model call).

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

turn = api("POST", "/api/v1/copilot/turn", {
    "message": "What is the current wave risk?",
    "scenario_name": "labor_constraint_wave_risk",
}, timeout=120)
if turn:
    CONVERSATION_ID = turn["conversation_id"]
    ASK_TURN_ID = turn["turn_id"]
    print(f"Conversation: {CONVERSATION_ID}")
    print(f"Intent:       {turn['intent']}  status={turn['status']}")
    print(f"Model:        {turn.get('model_id')}  role={turn.get('selected_role')}  rule={turn.get('routing_rule')}")
    print(f"Snapshot:     {turn.get('context_snapshot_id')}")
    print(f"\nAnswer:\n{turn.get('answer')}")

## Step 10 — Ask Copilot (ANALYZE)

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

body = {
    "message": "What should I do about the labor shortage?",
    "scenario_name": "labor_constraint_wave_risk",
}
if "CONVERSATION_ID" in globals():
    body["conversation_id"] = CONVERSATION_ID

turn = api("POST", "/api/v1/copilot/turn", body, timeout=180)
if turn:
    CONVERSATION_ID = turn["conversation_id"]
    RECOMMENDATIONS = turn.get("recommendations") or []
    print(f"Intent:   {turn['intent']}  status={turn['status']}")
    print(f"Severity: {turn.get('severity')}")
    print(f"Summary:  {turn.get('summary')}")
    print(f"Recommendations: {len(RECOMMENDATIONS)}")

## Step 11 — Inspect the OperationalContextSnapshot

The exact bounded context MAIW saw at decision time, captured before the model call. It is immutable —
LIVE changes do not alter it. In the UI: WORLD → **CONTEXT** tab.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

turn_id = globals().get("ASK_TURN_ID")
if not turn_id:
    # Fall back to the newest captured snapshot in this API process.
    listing = api("GET", "/api/v1/world/context/snapshots")
    snaps = (listing or {}).get("snapshots", [])
    turn_id = snaps[0]["turn_id"] if snaps else None

if not turn_id:
    print("No context snapshot captured yet — run a grounded Copilot turn (step 9) first.")
else:
    snap = api("GET", f"/api/v1/world/context/by-turn/{turn_id}")
    if snap:
        print(f"context_snapshot_id: {snap['context_snapshot_id']}")
        print(f"turn_id / trace_id:  {snap['turn_id']} / {snap['trace_id']}")
        print(f"focus:               {snap['focus_label']} ({snap['focus_entity_type']} {snap['focus_entity_id']})")
        print(f"entities / edges:    {snap['entity_count']} / {snap['relationship_count']}  truncated={snap['truncated']}")
        print(f"datapack_checksum:   {snap['datapack_checksum']}")
        print(f"captured_at:         {snap['captured_at']}")

## Steps 12-13 — SOP-Driven Agent and RecommendedAction

The OperationsCoordinationAgent follows the `operations_coordination.wave_risk_resolution` SOP
(`agents/sops/operations_coordination/`). Its output is a `RecommendedAction` — a semantic description of
the intervention (capability, target, rationale). It contains no MCP parameters; the governance layer
turns it into an `ActionProposal` in step 14.

If `MAIW_AGENT_RUNTIME=deep_agents` is set on the backend, the coordinator delegates to specialist
subagents through the Deep Agents runtime; the default is the deterministic runtime.

In [ ]:
recs = globals().get("RECOMMENDATIONS")
if not recs:
    print("No recommendations yet — run step 10 (ANALYZE) first.")
else:
    for i, rec in enumerate(recs, 1):
        print(f"[{i}] {rec.get('capability')} → {rec.get('target')}  priority={rec.get('priority')}  domain={rec.get('domain')}")
        print(f"    objective: {rec.get('objective')}")
        print(f"    rationale: {(rec.get('rationale') or '')[:200]}")

## Step 14 — Execute the Governed Action

**MAIW AUTHORITY BOUNDARY.** "Do it." is a Copilot **ACT** turn: it turns the first recommendation into an
`ActionProposal` and sends it to the DecisionEngine. Copilot never approves or executes on its own. When the
decision is `REQUIRES_HUMAN_APPROVAL`, the proposal waits in the approval queue for an operator.

The ACT turn also records the SOP-driven agent task (`agent_task_id`), shown below.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

if "CONVERSATION_ID" not in globals():
    print("Run steps 9-10 first: ACT resolves the recommendation from the ANALYZE turn in the same conversation.")
else:
    turn = api("POST", "/api/v1/copilot/turn", {
        "message": "Do it.",
        "conversation_id": CONVERSATION_ID,
        "scenario_name": "labor_constraint_wave_risk",
    }, timeout=120)
    if turn:
        print(f"Intent:            {turn['intent']}  status={turn['status']}")
        print(f"Decision outcome:  {turn.get('act_decision_outcome')}")
        print(f"Proposal ID:       {turn.get('act_proposal_id')}")
        print(f"Pending approval:  {turn.get('act_pending_approval_id')}")
        print(f"Execution status:  {turn.get('act_execution_status')}")
        agent_task_id = turn.get("agent_task_id")
        if agent_task_id:
            task = api("GET", f"/api/v1/agent-tasks/{agent_task_id}")
            if task:
                print(f"\nAgent task:  {task['task_id']}  agent={task['agent_id']}")
                print(f"SOP:         {task['sop_id']} v{task['sop_version']}  status={task['status']}")
                print(f"Steps done:  {task['completed_steps']}")

### Approve (human decision)

In the UI the operator approves on OPERATIONS (`http://localhost:3001/demo`) → APPROVE stage.
For this walkthrough the cell below plays the operator: it reads the approval queue from
`GET /api/v1/demo/status` and approves with `POST /api/v1/demo/approve` (`{"pending_id", "approved_by"}`).
Approval runs the execution through the ActionExecutor and MCP.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

status = api("GET", "/api/v1/demo/status")
pending = (status or {}).get("pending_approvals", [])
if not pending:
    print("No pending approvals — complete step 14 (ACT) first, or the decision did not require human approval.")
else:
    item = pending[0]
    print(f"Approving {item['pending_id']}: {item['capability']} → {item['target']} (risk={item.get('risk_level')})")
    result = api("POST", "/api/v1/demo/approve", {"pending_id": item["pending_id"], "approved_by": "notebook-operator"}, timeout=120)
    if result:
        print(f"ok={result.get('ok')}  status={result.get('status')}")
        print(f"execution_id: {result.get('execution_id')}")
        if result.get("reconciliation_required"):
            print("Outcome UNKNOWN — reconciliation required (no automatic retry).")

## Step 15 — Observe the LIVE Outcome

LIVE is the mutable runtime state; the DataPack checksum does not change. In the UI: WORLD → **LIVE**.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

live_world = api("GET", "/api/v1/world/live")
if live_world:
    print(f"Runtime:          {live_world['runtime_status']}  scenario={live_world['scenario_id']}")
    print(f"Summary:          {live_world['summary']}")
    print(f"Changed entities: {len(live_world['changed_entities'])}")
    for ent in live_world["changed_entities"][:10]:
        print(f"  {ent['entity_type']:7} {ent['entity_id']}: {[f['field'] for f in ent['changed_fields']]}")
    last = live_world.get("last_execution")
    if last:
        print(f"Last execution:   {last.get('execution_id')}  outcome={last.get('outcome')}  kpi_delta={last.get('kpi_delta')}")
    print(f"Base checksum:    {live_world['base_checksum']}")

## Step 16 — Ask Copilot "Did it work?" (OBSERVE_OUTCOME)

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

if "CONVERSATION_ID" not in globals():
    print("Run steps 9-14 first so the conversation has an ACT turn to assess.")
else:
    turn = api("POST", "/api/v1/copilot/turn", {
        "message": "Did it work?",
        "conversation_id": CONVERSATION_ID,
        "scenario_name": "labor_constraint_wave_risk",
    }, timeout=120)
    if turn:
        print(f"Intent:               {turn['intent']}  status={turn['status']}")
        print(f"Execution confirmed:  {turn.get('observe_execution_confirmed')}")
        print(f"Operational improved: {turn.get('observe_operational_improved')}")
        print(f"KPI delta:            {turn.get('observe_kpi_delta')}")
        print(f"\n{turn.get('observe_operational_summary') or turn.get('answer')}")

## Step 17 — Open the Model Gateway Evaluation Lab

The Lab is a read-only developer view of committed evaluation artifacts: `http://localhost:3001/models/lab`.
It cannot create proposals, invoke the DecisionEngine or call MCP writes; secrets are stripped from responses.

In [ ]:
assert "api" in globals(), "Run the API helper cell (Steps 5-6) first."

runs = api("GET", "/api/v1/model-lab/runs")
if runs is not None:
    print("Model Gateway Evaluation Lab — runs:")
    for run in runs:
        print(f"  {run['run_id']:4} {run['phase']:4} artifact_available={run['artifact_available']}  methodology_valid={run['methodology_valid']}")
    print("\nOpen in browser: http://localhost:3001/models/lab")

## Summary

| Step | What you did |
|------|--------------|
| 1    | Validated Python, Node.js and GPU environment |
| 2    | Installed requirements and all ten MAIW packages (editable) |
| 3    | Configured NVIDIA API key and model endpoints |
| 4    | Validated the committed DataPack (`dc47-demo-v1`, seed=42) |
| 5-6  | Started backend (uvicorn, Demo Mode) and frontend (npm) |
| 7    | Inspected the WORLD view (operational graph) |
| 8    | Activated the Wave 17 scenario (`labor_constraint_wave_risk`) |
| 9    | Asked Copilot about wave risk (ASK) |
| 10   | Asked Copilot for recommendations (ANALYZE) |
| 11   | Inspected the OperationalContextSnapshot |
| 12-13| Reviewed the SOP-driven RecommendedAction |
| 14   | Proposed via Copilot ACT and approved as the operator (authority boundary) |
| 15   | Observed the LIVE warehouse outcome |
| 16   | Asked Copilot whether it worked (OBSERVE_OUTCOME) |
| 17   | Explored the Model Gateway Evaluation Lab |

---

**Next steps:**
- [Developer Getting Started](../docs/developer/GETTING_STARTED.md)
- [Adding an Agent or Skill](../docs/developer/ADDING_AN_AGENT_OR_SKILL.md)
- [SOP Engine v2 Design](../docs/architecture/SOP_ENGINE_V2_DESIGN.md)
- [Model Gateway Evaluation](../docs/developer/MODEL_GATEWAY_EVALUATION.md)
- [Agent Runtime](../docs/architecture/AGENT_RUNTIME.md)
- [MAIW Glossary](../docs/GLOSSARY.md)